In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import sys, os
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from scr.transform.data_cleaning import (
    remove_unnecessary_cols, date_time_extract,
    plot_corr_heatmap, time_transform, iv_woe_all
)
from scr.transform.preprocessing import (
    train_dev_test, preprocess_features
)
df = pd.read_csv ("~/Documents/us-flight-project/data/bronze_data/merge.csv").drop (columns="Unnamed: 0")
df

,FlightDate,Day_Of_Week,Airline,Tail_Number,Dep_Airport,Dep_CityName,DepTime_label,Dep_Delay,Dep_Delay_Tag,Dep_Delay_Type,...,COUNTRY_dep,LATITUDE_dep,LONGITUDE_dep,IATA_CODE_arr,AIRPORT_arr,CITY_arr,STATE_arr,COUNTRY_arr,LATITUDE_arr,LONGITUDE_arr
0,2023-01-02,1,Endeavor Air,N605LR,BDL,"Hartford, CT",Morning,-3,0,Low <5min,...,USA,41.93887,-72.68323,LGA,LaGuardia Airport (Marine Air Terminal),New York,NY,USA,40.77724,-73.87261
1,2023-01-03,2,Endeavor Air,N605LR,BDL,"Hartford, CT",Morning,-5,0,Low <5min,...,USA,41.93887,-72.68323,LGA,LaGuardia Airport (Marine Air Terminal),New York,NY,USA,40.77724,-73.87261
2,2023-01-04,3,Endeavor Air,N331PQ,BDL,"Hartford, CT",Morning,-5,0,Low <5min,...,USA,41.93887,-72.68323,LGA,LaGuardia Airport (Marine Air Terminal),New York,NY,USA,40.77724,-73.87261
3,2023-01-05,4,Endeavor Air,N906XJ,BDL,"Hartford, CT",Morning,-6,0,Low <5min,...,USA,41.93887,-72.68323,LGA,LaGuardia Airport (Marine Air Terminal),New York,NY,USA,40.77724,-73.87261
4,2023-01-06,5,Endeavor Air,N337PQ,BDL,"Hartford, CT",Morning,-1,0,Low <5min,...,USA,41.93887,-72.68323,LGA,LaGuardia Airport (Marine Air Terminal),New York,NY,USA,40.77724,-73.87261
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6743399,2023-12-31,7,JetBlue Airways,N903JB,SJU,"San Juan, PR",Morning,4,1,Low <5min,...,USA,18.43942,-66.00183,JFK,John F. Kennedy International Airport (New Yor...,New York,NY,USA,40.63975,-73.77893
6743400,2023-12-31,7,JetBlue Airways,N535JB,MCO,"Orlando, FL",Evening,113,1,Hight >60min,...,USA,28.42889,-81.31603,SJU,Luis Muñoz Marín International Airport,San Juan,PR,USA,18.43942,-66.00183
6743401,2023-12-31,7,JetBlue Airways,N354JB,PHL,"Philadelphia, PA",Afternoon,-11,0,Low <5min,...,USA,39.87195,-75.24114,BOS,Gen. Edward Lawrence Logan International Airport,Boston,MA,USA,42.36435,-71.00518
6743402,2023-12-31,7,JetBlue Airways,N768JB,PBI,"West Palm Beach/Palm Beach, FL",Afternoon,-7,0,Low <5min,...,USA,26.68316,-80.09559,BDL,Bradley International Airport,Windsor Locks,CT,USA,41.93887,-72.68323


In [ ]:
unnecessary_cols = [
    "Tail_Number", "Dep_CityName", "Arr_CityName", "Dep_Delay",
    "Dep_Delay_Type", "Arr_Delay", "Arr_Delay_Type",
    "Delay_Carrier", "Delay_Weather", "Delay_NAS", "Delay_Security",
    "Delay_LastAircraft", "time_dep", "time_arr",  "Model",
    "airport_id_arr", "airport_id_dep", "COUNTRY_dep", "COUNTRY_arr",
    "IATA_CODE_arr", "IATA_CODE_dep",
    "AIRPORT_arr", "AIRPORT_dep",   # Deep learning không xoá
    "LATITUDE_dep", "LONGITUDE_dep", "LATITUDE_arr", "LONGITUDE_arr",
    "CITY_dep", "STATE_dep",
    "CITY_arr", "STATE_arr"
]
df = remove_unnecessary_cols (df, unnecessary_cols)

Deleted 31 columns.


In [ ]:
df

,FlightDate,Day_Of_Week,Airline,Dep_Airport,DepTime_label,Dep_Delay_Tag,Arr_Airport,Flight_Duration,Distance_type,Manufacturer,...,wspd_dep,pres_dep,tavg_arr,tmin_arr,tmax_arr,prcp_arr,snow_arr,wdir_arr,wspd_arr,pres_arr
0,2023-01-02,1,Endeavor Air,BDL,Morning,0,LGA,56,Short Haul >1500Mi,CANADAIR REGIONAL JET,...,3.2,1019.1,11.4,9.4,13.3,0.5,0.0,265.0,6.8,1019.7
1,2023-01-03,2,Endeavor Air,BDL,Morning,0,LGA,62,Short Haul >1500Mi,CANADAIR REGIONAL JET,...,3.6,1015.2,11.2,8.0,14.0,9.6,0.0,165.0,7.9,1014.6
2,2023-01-04,3,Endeavor Air,BDL,Morning,0,LGA,49,Short Haul >1500Mi,CANADAIR REGIONAL JET,...,7.2,1011.1,14.5,9.4,19.0,9.9,0.0,208.0,8.9,1010.6
3,2023-01-05,4,Endeavor Air,BDL,Morning,0,LGA,54,Short Haul >1500Mi,CANADAIR REGIONAL JET,...,13.7,1014.8,8.7,7.0,10.0,0.4,0.0,54.0,8.1,1014.1
4,2023-01-06,5,Endeavor Air,BDL,Morning,0,LGA,50,Short Haul >1500Mi,CANADAIR REGIONAL JET,...,5.8,1016.1,6.8,3.9,8.9,7.5,0.0,317.0,8.8,1016.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6743399,2023-12-31,7,JetBlue Airways,SJU,Morning,1,JFK,219,Medium Haul <3000Mi,AIRBUS,...,8.3,1017.8,5.1,3.3,6.1,0.0,0.0,297.0,19.4,1014.5
6743400,2023-12-31,7,JetBlue Airways,MCO,Evening,1,SJU,162,Short Haul >1500Mi,AIRBUS,...,5.6,1023.8,25.8,23.9,28.3,1.5,0.0,37.0,8.3,1017.8
6743401,2023-12-31,7,JetBlue Airways,PHL,Afternoon,0,BOS,73,Short Haul >1500Mi,EMBRAER,...,9.7,1015.5,2.6,0.6,5.0,0.0,0.0,316.0,13.0,1012.3
6743402,2023-12-31,7,JetBlue Airways,PBI,Afternoon,0,BDL,158,Short Haul >1500Mi,AIRBUS,...,4.0,1023.3,2.7,1.7,3.9,0.0,0.0,312.0,14.0,1013.3


In [ ]:
df = time_transform (df, date_col="FlightDate")
extract_cols = [
    "month", "day"
]
df = date_time_extract (df, org_col="FlightDate", types=extract_cols)

Transform FlightDate to datetime
Create month column.
Create day column.
Done


In [ ]:
df.drop(columns="FlightDate", inplace=True)

In [ ]:
X = df.drop(columns="Dep_Delay_Tag")
y = df["Dep_Delay_Tag"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

print(f"\nNumerical columns for scaling: {numerical_cols}")
print(f"Categorical columns for One-Hot Encoding: {categorical_cols}")

# Define preprocessing pipelines
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # In case any NaNs remain
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])



Numerical columns for scaling: ['Day_Of_Week', 'Flight_Duration', 'Aicraft_age', 'tavg_dep', 'tmin_dep', 'tmax_dep', 'prcp_dep', 'snow_dep', 'wdir_dep', 'wspd_dep', 'pres_dep', 'tavg_arr', 'tmin_arr', 'tmax_arr', 'prcp_arr', 'snow_arr', 'wdir_arr', 'wspd_arr', 'pres_arr', 'month', 'day']
Categorical columns for One-Hot Encoding: ['Airline', 'Dep_Airport', 'DepTime_label', 'Arr_Airport', 'Distance_type', 'Manufacturer']


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'
)
print("\nApplying preprocessing to datasets...")
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert to dense arrays (if OneHotEncoder sparse)
X_train_processed = X_train_processed.toarray()
X_test_processed = X_test_processed.toarray()


Applying preprocessing to datasets...


In [ ]:
print(f"Dimensions of X_train_processed (after toarray): {X_train_processed.shape}")
print(f"Dimensions of X_test_processed (after toarray): {X_test_processed.shape}")

# Reshape for LSTM input (samples, timesteps, features)
timesteps = 1
X_train_lstm = X_train_processed.reshape(X_train_processed.shape[0], timesteps, X_train_processed.shape[1])
X_test_lstm = X_test_processed.reshape(X_test_processed.shape[0], timesteps, X_test_processed.shape[1])

Dimensions of X_train_processed (after toarray): (5394723, 748)
Dimensions of X_test_processed (after toarray): (1348681, 748)


In [ ]:
print(f"Final dimensions for LSTM (X_train_lstm): {X_train_lstm.shape}")
print(f"Final dimensions for LSTM (X_test_lstm): {X_test_lstm.shape}")

# Save preprocessor and processed data
joblib.dump(preprocessor, 'preprocessor.pkl')
print("\nPreprocessor saved as 'preprocessor.pkl'")

np.save('/Users/kittnguyen/Documents/us-flight-project/data/deep_learning_data/train/X_train_lstm.npy', X_train_lstm)
np.save('/Users/kittnguyen/Documents/us-flight-project/data/deep_learning_data/test/X_test_lstm.npy', X_test_lstm)
np.save('/Users/kittnguyen/Documents/us-flight-project/data/deep_learning_data/train/y_train.npy', y_train)
np.save('/Users/kittnguyen/Documents/us-flight-project/data/deep_learning_data/test/y_test.npy', y_test)
print("Processed data saved and ready for model training")

print("\nPreprocessing completed successfully and data ready for modeling")

Final dimensions for LSTM (X_train_lstm): (5394723, 1, 748)
Final dimensions for LSTM (X_test_lstm): (1348681, 1, 748)

Preprocessor saved as 'preprocessor.pkl'


OSError: 4035252804 requested and 3489659392 written

: 